# Train the emergency-vehicle model on a free Colab GPU

This trains `models/emergency_vehicle.pt` (ambulance, fire truck, police car) with the cleaned dataset in the repository, scores it on the validation **and test** images, then lets you download it.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Training takes roughly 20 to 40 minutes for 60 epochs. Colab sessions are temporary, so download the result when it finishes.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: use Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
# The cleaned dataset is on main.
BRANCH = 'main'
!git clone --branch $BRANCH https://github.com/aditya182005/Real_Time_AI_Traffic_Monitoring_Violation_Enforcement.git traffic-agent
%cd traffic-agent
!pip -q install ultralytics "supervision>=0.20,<0.31"

In [ ]:
# Confirms the dataset is complete before spending GPU time.
import sys; sys.path.insert(0, 'scripts')
from train_emergency import check_dataset
print(check_dataset())

In [ ]:
# 60 epochs is a good start. --batch -1 lets Ultralytics pick the batch size.
!python scripts/train_emergency.py --epochs 60 --device 0 --batch -1 --report emergency_scores.json

The last lines show **mAP@0.5, precision and recall** on the validation images and, separately, the **test** images the model never trained on (with each class listed). Quote the test numbers in your report. As a rough guide: above 0.8 is good, 0.5 to 0.8 is usable with misses and false alarms, below 0.5 needs more training or better labels.

These photos are mostly close-ups, so real CCTV footage will score lower.

In [ ]:
# Sanity check on a real photo from the repo
from ultralytics import YOLO
model = YOLO('models/emergency_vehicle.pt')
results = model('data/emergency_vehicle/images/val/' + __import__('os').listdir('data/emergency_vehicle/images/val')[0])
results[0].show()

In [ ]:
from google.colab import files
files.download('models/emergency_vehicle.pt')

## Then, on your own computer
1. Put the downloaded file at `models/emergency_vehicle.pt` in the project.
2. Run `python check_setup.py --full`.
3. In the app, the **Emergency vehicles** option on *Analyze footage* and *Live camera* is now available.

The weights are git-ignored on purpose. For a server, mount the `models` folder (see `docker-compose.yml`).